# CICIoV2024 – Deneyler (AS2 ve AS3)

**AS2 – Makine öğrenmesi gerçekten gerekli mi?**
Literatürdeki rastgele %80/20 bölme protokolünde makine öğrenmesi modellerini iki *önemsiz* temel çizgiyle karşılaştırır:
- **Birebir eşleşme (ezber) tablosu:** "Bu çerçeveyi eğitimde gördüm mü? Gördüysem etiketi neydi?"
- **Yalnızca CAN ID:** Sadece ID sütununa bakan karar ağacı.

Bu basit yöntemler modellerle aynı skoru alırsa, yüksek başarı ezberden kaynaklanıyor demektir.

**ID analizi – Tehdit modeli gerçekçi mi?** Saldırıların kullandığı CAN ID'leri normal trafikte de geçiyor mu?

**AS3 – Görülmemiş saldırı tespit edilebilir mi?** (sızıntısız)
- **3a – Tek sınıflı anomali tespiti:** Sadece benzersiz *normal* çerçevelerle eğitim; test kümesinde hiç görülmemiş saldırı imzaları.
- **3b – Bir saldırı türünü dışarıda bırakma (leave-one-attack-out):** Bir saldırı türü eğitimden tamamen çıkarılır, model onu tespit edebiliyor mu?

Her iki deney **iki öznitelik kümesiyle** yapılır: *ID + veri baytları* ve *yalnızca veri baytları*. İkincisi, tespitin ID kısayoluna dayanıp dayanmadığını gösterir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CICIoV2024/decimal'   # <-- kendi klasörünle değiştir
OUT_DIR = '/content/drive/MyDrive/CICIoV2024/sonuclar'   # sonuç tabloları buraya kaydedilir
SEEDS = [0, 1, 2, 3, 4]
SAMPLE_FRAC = 1.0   # AS2 çok uzun sürerse ya da bellek yetmezse 0.3 gibi bir değer ver

In [ ]:
import glob, os, time, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
os.makedirs(OUT_DIR, exist_ok=True)

frames = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))]
df = pd.concat(frames, ignore_index=True)
df.columns = [c.strip() for c in df.columns]

label_cols = [c for c in df.columns if c.lower() in ('label', 'category', 'specific_class')]
TARGET = 'specific_class' if 'specific_class' in df.columns else label_cols[-1]
feature_cols = [c for c in df.columns if c not in label_cols]
ID_COL = next(c for c in feature_cols if c.upper() == 'ID')
PAYLOAD_COLS = [c for c in feature_cols if c != ID_COL]
FEATURE_SETS = {'ID+payload': feature_cols, 'payload': PAYLOAD_COLS}
BENIGN = 'BENIGN'
print(df.shape, '| hedef:', TARGET, '| öznitelikler:', feature_cols)

## Ortak araçlar

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, matthews_corrcoef, accuracy_score, roc_auc_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

class ExactMatch:
    """Ezber temel çizgisi: eğitimde birebir görülen vektörün etiketini döndürür, görülmeyene çoğunluk sınıfını verir."""
    def fit(self, X, y):
        d = pd.DataFrame(X).assign(_y=y).drop_duplicates(subset=list(range(X.shape[1])))
        self.table = dict(zip(map(tuple, d.iloc[:, :-1].to_numpy()), d['_y']))
        vals, cnt = np.unique(y, return_counts=True)
        self.default = vals[cnt.argmax()]
        return self
    def predict(self, X):
        return np.array([self.table.get(tuple(r), self.default) for r in X])

class IDOnly:
    """Yalnızca CAN ID sütununu kullanan karar ağacı."""
    def __init__(self, id_idx): self.id_idx = id_idx
    def fit(self, X, y):
        self.m = DecisionTreeClassifier(random_state=0).fit(X[:, [self.id_idx]], y); return self
    def predict(self, X):
        return self.m.predict(X[:, [self.id_idx]])

def size_kb(model):
    try: return len(pickle.dumps(model)) / 1024
    except Exception: return np.nan

def ms(a): return f'{np.mean(a):.4f} ± {np.std(a):.4f}'
print('LightGBM:', HAS_LGB)

## AS2 – Rastgele bölmede modeller ve önemsiz temel çizgiler
Literatür protokolü: tüm satırlar (tekrarlar dahil), katmanlı rastgele %80/20 bölme. Her tohum için ayrı bölme.

In [ ]:
data = df if SAMPLE_FRAC >= 1 else df.groupby(TARGET, group_keys=False).apply(
    lambda g: g.sample(frac=SAMPLE_FRAC, random_state=0))
X_all = data[feature_cols].to_numpy()
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_all = le.fit_transform(data[TARGET])   # etiketler sayıya çevrilir (MLP uyumluluğu için)
print('Sınıf kodları:', dict(enumerate(le.classes_)))
id_idx = feature_cols.index(ID_COL)

def as2_models(seed):
    m = {
        'Ezber tablosu (temel çizgi)': ExactMatch(),
        'Yalnızca CAN ID (temel çizgi)': IDOnly(id_idx),
        'Lojistik regresyon': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        'Karar ağacı': DecisionTreeClassifier(random_state=seed),
        'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
        'MLP': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                                             early_stopping=True, random_state=seed)),
    }
    if HAS_LGB:
        m['LightGBM'] = lgb.LGBMClassifier(n_estimators=200, random_state=seed, verbose=-1)
    return m

rows = []
for seed in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=seed)
    for name, model in as2_models(seed).items():
        t0 = time.time(); model.fit(Xtr, ytr); t_fit = time.time() - t0
        t0 = time.time(); pred = model.predict(Xte); t_pred = time.time() - t0
        rows.append({'model': name, 'seed': seed,
                     'accuracy': accuracy_score(yte, pred),
                     'macro_F1': f1_score(yte, pred, average='macro'),
                     'MCC': matthews_corrcoef(yte, pred),
                     'egitim_s': t_fit,
                     'cikarim_us_ornek': t_pred / len(Xte) * 1e6,
                     'boyut_KB': size_kb(model)})
        print(f'seed={seed}  {name:32s} macroF1={rows[-1]["macro_F1"]:.4f}  ({t_fit:.1f} s)')

as2 = pd.DataFrame(rows)
as2.to_csv(os.path.join(OUT_DIR, 'as2_ham.csv'), index=False)
as2_summary = as2.groupby('model', sort=False).agg(
    accuracy=('accuracy', ms), macro_F1=('macro_F1', ms), MCC=('MCC', ms),
    cikarim_us_ornek=('cikarim_us_ornek', 'mean'), boyut_KB=('boyut_KB', 'mean')).round(2)
as2_summary.to_csv(os.path.join(OUT_DIR, 'as2_ozet.csv'))
as2_summary

## ID analizi – Saldırı ID'leri normal trafikte geçiyor mu?
Gerçekçi bir sahtecilik saldırısında saldırgan **meşru bir ID** kullanıp veri baytlarını değiştirir. Saldırı ID'si normal trafikte hiç yoksa, basit bir ID beyaz listesi saldırıyı yakalar.

In [ ]:
U = df.drop_duplicates(subset=feature_cols + [TARGET]).reset_index(drop=True)
counts = df.groupby(feature_cols + [TARGET]).size().rename('tekrar_sayisi').reset_index()
U = U.merge(counts, on=feature_cols + [TARGET], how='left')
ben_U = U[U[TARGET] == BENIGN].reset_index(drop=True)
att_U = U[U[TARGET] != BENIGN].reset_index(drop=True)
benign_ids = set(ben_U[ID_COL])

id_rows = []
for cls, g in att_U.groupby(TARGET):
    for cid, gg in g.groupby(ID_COL):
        id_rows.append({'saldiri': cls, 'CAN_ID': cid, 'ID_hex': hex(int(cid)),
                        'benzersiz_cerceve': len(gg), 'toplam_satir': int(gg['tekrar_sayisi'].sum()),
                        'normal_trafikte_var_mi': cid in benign_ids,
                        'bu_ID_ile_normal_cerceve': int((ben_U[ID_COL] == cid).sum())})
id_table = pd.DataFrame(id_rows)
id_table.to_csv(os.path.join(OUT_DIR, 'id_analizi.csv'), index=False)
print('Normal trafikteki farklı ID sayısı:', len(benign_ids))
print('Saldırılarda kullanılan farklı ID sayısı:', att_U[ID_COL].nunique())
id_table

## AS3a – Tek sınıflı anomali tespiti (yalnızca normal trafikle eğitim)
- Benzersiz normal çerçeveler %80 eğitim / %20 test olarak bölünür (her tohumda farklı).
- Test: ayrılan normal çerçeveler + **tüm benzersiz saldırı çerçeveleri** (hiçbiri eğitimde yok).
- Eşik, eğitim skorlarının 99. yüzdeliğidir (hedef ≈ %1 yanlış alarm).
- **Tespit oranı (çerçeve)**: benzersiz saldırı çerçevelerinin yakalanan oranı.
- **Tespit oranı (trafik ağırlıklı)**: her çerçeve orijinal veri setinde kaç kez geçiyorsa o kadar ağırlıkla.

In [ ]:
def oc_detectors(seed):
    return {
        'ID beyaz listesi (kural)': None,
        'Isolation Forest': IsolationForest(n_estimators=200, random_state=seed),
        'One-Class SVM': make_pipeline(StandardScaler(), OneClassSVM(nu=0.01, gamma='scale')),
        'LOF': make_pipeline(StandardScaler(), LocalOutlierFactor(n_neighbors=20, novelty=True)),
        'Otokodlayıcı': 'AE',
    }

def anomaly_scores(name, det, Xtr, Xte, feats, seed):
    """Büyük skor = daha anormal. Eğitim ve test skorlarını döndürür."""
    if det is None:
        if ID_COL not in feats: return None, None
        k = feats.index(ID_COL); allowed = set(Xtr[:, k])
        f = lambda X: np.array([0.0 if v in allowed else 1.0 for v in X[:, k]])
        return f(Xtr), f(Xte)
    if det == 'AE':
        sc = StandardScaler().fit(Xtr)
        A, B = sc.transform(Xtr), sc.transform(Xte)
        ae = MLPRegressor(hidden_layer_sizes=(16, 4, 16), max_iter=2000, random_state=seed).fit(A, A)
        return ((ae.predict(A) - A) ** 2).mean(1), ((ae.predict(B) - B) ** 2).mean(1)
    det.fit(Xtr)
    return -det.score_samples(Xtr), -det.score_samples(Xte)

rows, per_class_rows = [], []
for fs_name, feats in FEATURE_SETS.items():
    for seed in SEEDS:
        btr, bte = train_test_split(ben_U, test_size=0.2, random_state=seed)
        test = pd.concat([bte, att_U], ignore_index=True)
        y_bin = (test[TARGET] != BENIGN).astype(int).to_numpy()
        Xtr, Xte = btr[feats].to_numpy().astype(float), test[feats].to_numpy().astype(float)
        for name, det in oc_detectors(seed).items():
            s_tr, s_te = anomaly_scores(name, det, Xtr, Xte, feats, seed)
            if s_tr is None: continue
            thr = 0.5 if det is None else np.percentile(s_tr, 99)
            flag = s_te > thr
            att_mask = y_bin == 1
            w = test.loc[att_mask, 'tekrar_sayisi'].to_numpy()
            rows.append({'oznitelik': fs_name, 'yontem': name, 'seed': seed,
                         'AUROC': roc_auc_score(y_bin, s_te),
                         'tespit_orani_cerceve': flag[att_mask].mean(),
                         'tespit_orani_trafik': (flag[att_mask] * w).sum() / w.sum(),
                         'yanlis_alarm_orani': flag[~att_mask].mean()})
            for cls in att_U[TARGET].unique():
                m = (test[TARGET] == cls).to_numpy()
                per_class_rows.append({'oznitelik': fs_name, 'yontem': name, 'seed': seed,
                                       'saldiri': cls, 'tespit_orani': flag[m].mean()})

as3a = pd.DataFrame(rows)
as3a.to_csv(os.path.join(OUT_DIR, 'as3a_ham.csv'), index=False)
as3a_summary = as3a.groupby(['oznitelik', 'yontem'], sort=False).agg(
    AUROC=('AUROC', ms), tespit_cerceve=('tespit_orani_cerceve', ms),
    tespit_trafik=('tespit_orani_trafik', ms), yanlis_alarm=('yanlis_alarm_orani', ms))
as3a_summary.to_csv(os.path.join(OUT_DIR, 'as3a_ozet.csv'))
as3a_summary

In [ ]:
as3a_cls = (pd.DataFrame(per_class_rows)
            .groupby(['oznitelik', 'yontem', 'saldiri'], sort=False)['tespit_orani'].mean()
            .unstack('saldiri').round(3))
as3a_cls.to_csv(os.path.join(OUT_DIR, 'as3a_sinif_bazinda.csv'))
as3a_cls

## AS3b – Bir saldırı türünü dışarıda bırakma (denetimli, ikili sınıflandırma)
Her turda bir saldırı türü eğitimden **tamamen** çıkarılır. Model, normal + diğer saldırı türleriyle (ikili etiket: normal/saldırı) eğitilir ve dışarıda bırakılan türü tespit edip edemediği ölçülür. Benzersiz satırlar kullanıldığı için sızıntı yoktur.

In [ ]:
attack_types = sorted(att_U[TARGET].unique())
rows = []
for fs_name, feats in FEATURE_SETS.items():
    for seed in SEEDS:
        btr, bte = train_test_split(ben_U, test_size=0.2, random_state=seed)
        for held in attack_types:
            tr = pd.concat([btr, att_U[att_U[TARGET] != held]], ignore_index=True)
            te = pd.concat([bte, att_U[att_U[TARGET] == held]], ignore_index=True)
            ytr = (tr[TARGET] != BENIGN).astype(int); yte = (te[TARGET] != BENIGN).astype(int).to_numpy()
            clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                         n_jobs=-1, random_state=seed).fit(tr[feats], ytr)
            pred = clf.predict(te[feats])
            rows.append({'oznitelik': fs_name, 'seed': seed, 'disarida_birakilan': held,
                         'tespit_orani': pred[yte == 1].mean(),
                         'yanlis_alarm_orani': pred[yte == 0].mean()})

as3b = pd.DataFrame(rows)
as3b.to_csv(os.path.join(OUT_DIR, 'as3b_ham.csv'), index=False)
as3b_summary = as3b.groupby(['oznitelik', 'disarida_birakilan']).agg(
    tespit_orani=('tespit_orani', ms), yanlis_alarm=('yanlis_alarm_orani', ms))
as3b_summary.to_csv(os.path.join(OUT_DIR, 'as3b_ozet.csv'))
as3b_summary

## Bana gönderilecek çıktılar
`sonuclar` klasöründeki şu dosyaları ya da ekran görüntülerini gönder:
`as2_ozet.csv`, `id_analizi.csv`, `as3a_ozet.csv`, `as3a_sinif_bazinda.csv`, `as3b_ozet.csv`